In [ ]:
!pip install langchain_groq langchain_core

/content/drive/MyDrive/datasets/tips.csvCollecting langchain_groq
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.3 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from google.colab import userdata
import os


def generate_report(dataframe, groq_api_key):
    # Initialize Groq API
    chat_groq = ChatGroq(temperature=0, groq_api_key=groq_api_key, model_name="openai/gpt-oss-120b")

    # Create a prompt template for generating the report
    prompt_template = PromptTemplate(
        input_variables=["data_summary", "data_sample"],
        template="""
### RETAIL DATA ANALYSIS REQUEST:
You are an expert **Retail Business Analyst** with deep experience in sales performance, inventory optimization, customer behavior, profitability, and merchandising strategy.

Analyze the provided retail dataset (e.g. transactions, sales, products, customers, stores, dates, quantities, prices, categories, etc.) and generate a comprehensive, actionable retail performance report.

### RETAIL DATA SUMMARY:
{data_summary}

### RETAIL DATA SAMPLE:
{data_sample}

### INSTRUCTIONS:
Act as a senior retail analyst preparing a report for store/chain management or category buyers. Use only the information available in {data_summary} for any numerical/statistical claims — do not assume or invent missing metrics.

Structure the report professionally using the following sections and markdown formatting:

## 1. Executive Summary
- 3–5 bullet points with the most important findings and business impact
- Highlight top-line revenue, profitability signals, strongest/weakest areas

## 2. Sales Performance Overview
- Total revenue, number of transactions, average transaction value (ATV / basket size)
- Sales trends over time (daily/weekly/monthly/seasonal patterns if dates are available)
- Top-selling products, categories, brands
- Underperforming products/categories (slow movers)

## 3. Customer & Transaction Insights
- Average items per transaction
- Repeat purchase / customer frequency signals (if customer IDs or timestamps allow inference)
- Distribution of transaction values (high-value vs. low-value baskets)
- Any visible customer segmentation patterns (if relevant fields exist)

## 4. Product & Category Performance
- Top performers by revenue, units sold, margin contribution (if price/cost available)
- Sell-through estimates or fast/slow-moving items
- Product assortment observations (breadth, depth, concentration risk)

## 5. Profitability & Margin Indicators
- Estimated gross margin trends (if unit price & cost present)
- Impact of discounting / promotions (if discount or promo fields exist)
- GMROI-like signals or inventory efficiency hints (if quantities & time periods allow)

## 6. Inventory & Operational Signals
- Any signs of stock-outs, overstock, or slow turnover (based on quantities sold vs. time)
- SKU-level concentration (Pareto: 80/20 rule observations)

## 7. Key Trends & Opportunities
- Emerging patterns, seasonality, day-of-week / time-of-day effects
- Cross-category opportunities or bundling potential
- Risks (declining categories, high return items if returns present)

## 8. Actionable Recommendations
- 5–8 prioritized, concrete recommendations for buyers, store managers, marketing, or operations
- Focus on quick wins vs. longer-term initiatives

## 9. Conclusion
- One-paragraph summary of overall business health and biggest levers for improvement

Use clear headings, sub-headings, bullet points, bold key numbers/metrics, and business language. Be specific with numbers from the data summary whenever possible. Keep the tone professional, confident, and commercially oriented — focus on decisions that drive sales, margin, and inventory productivity.
"""
    )

    # Prepare data summary and sample for the prompt
    data_summary = dataframe.describe(include='all').to_json()
    data_sample = dataframe.head().to_json()

    # Create the complete prompt
    prompt = prompt_template.format(data_summary=data_summary, data_sample=data_sample)

    # Call Groq API to generate the report
    response = chat_groq.invoke(prompt)  # Ensure correct method is used

    # Accessing the content of the response
    # Check the response structure and update this line accordingly
    report = response.content if hasattr(response, 'content') else "No report generated."

    return report

In [ ]:
# file_path = input("Enter path to CSV file: ").strip()
file_path = '/content/drive/MyDrive/datasets/tips.csv'

if not file_path:
    print("No file path provided.")

if not os.path.isfile(file_path):
    print(f"File not found: {file_path}")

if not file_path.lower().endswith('.csv'):
    print("File must be a .csv file")

try:
    df = pd.read_csv(file_path)
    print(f"Read successfully — shape: {df.shape}\n")
except pd.errors.EmptyDataError:
    print("CSV file is empty")

except pd.errors.ParserError:
    print("Cannot parse this file as CSV (wrong format?)")

except UnicodeDecodeError:
    print("File encoding problem — try saving as UTF-8")

except Exception as e:
    print(f"Failed to read CSV: {type(e).__name__}: {e}")


groq_api_key = userdata.get("GROQ_API_KEY")

if not groq_api_key:
    print("Warning: GROQ_API_KEY environment variable not found")
    groq_api_key = input("Enter Groq API key (or press Enter to skip): ").strip()
    if not groq_api_key:
        print("No API key → cannot generate report.")


try:
    report = generate_report(df, groq_api_key)
    print("\n" + "="*70)
    print(report)
    print("="*70 + "\n")
except Exception as e:
    print(f"Report generation failed: {type(e).__name__}")
    print(str(e))

Read successfully — shape: (244, 7)


## 1. Executive Summary
- **Total sales:** ~ **$4,828** across **244 transactions** (average ticket **$19.79**).  
- **Revenue concentration:** **Saturday** accounts for **≈ 35 %** of all sales (87 transactions) and **Dinner** for **≈ 72 %** (176 transactions).  
- **Basket size:** Average of **2.57 items** per ticket, translating to an average unit price of **≈ $7.70**.  
- **Tip margin proxy:** Tips average **$3.00**, representing **≈ 15 %** of the total bill – a useful indicator of service‑related profitability.  
- **Customer profile:** Predominantly **Male (64 %)** and **non‑smokers (62 %)**; these groups drive the bulk of traffic and revenue.

---

## 2. Sales Performance Overview  

| Metric | Value |
|--------|-------|
| **Total transactions** | 244 |
| **Total revenue (sum *total_bill*)** | **≈ $4,828** |
| **Average transaction value (ATV)** | **$19.79** |
| **Average items per transaction (size)** | **2.57** |
| **Average unit price** | 